# AresSim algorithm pipeline smoke test

Use this notebook to verify every baseline or future learned policy through the same public pipeline:

1. resolve the local AresSim package;
2. create and seed the framework-neutral environment;
3. validate the policy against the declared observation, action, and mask spaces;
4. collect a bounded episode;
5. save and validate an `aresim.trajectory.v1` dataset;
6. compare the in-memory, recorded, and deterministic repeat episodes.

The backend requirements are declared in `engine/pyproject.toml`. From the repository root, run `python3 -m venv engine/.venv`, then `engine/.venv/bin/python -m pip install -e './engine[dev,env,notebook]'`, and finally `engine/.venv/bin/python -m ipykernel install --user --name aresim --display-name "AresSim (.venv)"`. Select `AresSim (.venv)` as this notebook's kernel. The default `random_valid` policy samples only legal actions. Use `random` when intentionally testing invalid-action handling.

In [9]:
from __future__ import annotations

import sys
from collections.abc import Mapping, Sequence
from dataclasses import asdict
from pathlib import Path

def find_repository_root(start: Path) -> Path:
    """Find the checkout containing the local engine package."""
    for candidate in (start, *start.parents):
        if (candidate / "engine" / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("run this notebook from inside an AresSim checkout")


REPOSITORY_ROOT = find_repository_root(Path.cwd().resolve())
ENGINE_DIRECTORY = REPOSITORY_ROOT / "engine"
if str(ENGINE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(ENGINE_DIRECTORY))

try:
    import numpy as np

    from aresim.defaults import DEFAULT_ENVIRONMENT_CONFIG
    from aresim.factory import make_agent, make_env
    from aresim.training import (
        EpisodeSpec,
        RolloutConfig,
        RolloutRunner,
        TrajectoryWriter,
        iter_trajectory_episodes,
        validate_trajectory_dataset,
        validate_trajectory_episode,
    )
except ModuleNotFoundError as error:
    raise RuntimeError(
        "This notebook kernel is missing AresSim's RL dependencies. From the repository root run:\n"
        "engine/.venv/bin/python -m pip install -e './engine[dev,env,notebook]'\n"
        "Then select the 'AresSim (.venv)' kernel."
    ) from error

REPOSITORY_ROOT

PosixPath('/Users/shanmukh/Desktop/Projects/AresSim')

## Test parameters

Keep environment and policy seeds independent. Change `POLICY` to another registered name or an object implementing `aresim.algorithms.Agent`. For a custom registered policy or component composition, assign its registry to `REGISTRY`. The dataset path is intentionally immutable: change `DATASET_ID` before rerunning instead of overwriting an existing artifact.

In [10]:
AGENT_NAME = "scripted"  # random | random_valid | wait | scripted
ENVIRONMENT_SEED = 1447
AGENT_SEED = 9001
MAX_EPISODE_STEPS = 256
EPISODE_ID = f"{AGENT_NAME}-smoke-000"
DATASET_ID = f"{AGENT_NAME}-env{ENVIRONMENT_SEED}-agent{AGENT_SEED}-trajectory-v4"
COMPRESSION = "none"

ENVIRONMENT_CONFIG = DEFAULT_ENVIRONMENT_CONFIG
REGISTRY = None
POLICY = AGENT_NAME
OUTPUT_DIRECTORY = REPOSITORY_ROOT / "datasets" / DATASET_ID

## Create the environment and validate the policy boundary

This probe performs a reset but does not step the episode used for recording. The rollout runner creates its own environment and resets the policy with the same explicit seeds.

In [11]:
environment = make_env(
    ENVIRONMENT_CONFIG,
    registry=REGISTRY,
    max_episode_steps=MAX_EPISODE_STEPS,
)
initial = environment.reset(seed=ENVIRONMENT_SEED)
probe_agent = make_agent(POLICY, ENVIRONMENT_CONFIG, registry=REGISTRY) if isinstance(POLICY, str) else POLICY
probe_agent.reset(AGENT_SEED)
probe_action = probe_agent.act(initial.observation, initial.action_mask.copy())

assert environment.observation_space.contains(initial.observation)
assert environment.action_mask_space.contains(initial.action_mask)
assert environment.action_space.contains(probe_action)
assert probe_agent.action_schema == initial.info["action_schema"]
assert probe_agent.observation_schema in {None, initial.info["observation_schema"]}

{
    "policy_id": probe_agent.policy_id,
    "scenario_id": initial.info["scenario_id"],
    "task_id": initial.info["task_id"],
    "observation_schema": initial.info["observation_schema"],
    "action_schema": initial.info["action_schema"],
    "reward_profile": initial.info["reward_profile"],
    "initial_checksum": initial.info["state_checksum"],
    "probe_action": probe_action,
}

{'policy_id': 'aresim.agent.scripted.v1',
 'scenario_id': 'phase1_default_v1',
 'task_id': 'phase1_open_exploration_v1',
 'observation_schema': 'aresim.obs.local.v1',
 'action_schema': 'aresim.action.rover.v1',
 'reward_profile': 'aresim.reward.shaped_train.v1',
 'initial_checksum': '2013e15fedee6df62585eca181c36063b7750cc598ec0bb00527eda9ec3da124',
 'probe_action': 7}

## Collect and save one episode

`TrajectoryWriter` refuses to replace an existing dataset. This protects previously validated samples and makes accidental notebook reruns visible.

In [12]:
if OUTPUT_DIRECTORY.exists():
    raise FileExistsError(
        f"{OUTPUT_DIRECTORY} already exists; change DATASET_ID to create a new immutable dataset"
    )

rollout_config = RolloutConfig(
    episodes=(
        EpisodeSpec(
            episode_id=EPISODE_ID,
            environment_seed=ENVIRONMENT_SEED,
            agent_seed=AGENT_SEED,
        ),
    ),
    max_episode_steps=MAX_EPISODE_STEPS,
)
writer = TrajectoryWriter(
    OUTPUT_DIRECTORY,
    dataset_id=DATASET_ID,
    compression=COMPRESSION,
)
rollout = RolloutRunner(
    rollout_config,
    POLICY,
    environment_config=ENVIRONMENT_CONFIG,
    registry=REGISTRY,
).run(writer)

assert rollout.artifact_manifest == str((OUTPUT_DIRECTORY / "manifest.json").resolve())
[asdict(summary) for summary in rollout.summaries]

[{'episode_id': 'scripted-smoke-000',
  'policy_id': 'aresim.agent.scripted.v1',
  'scenario_id': 'phase1_default_v1',
  'environment_seed': 1447,
  'agent_seed': 9001,
  'length': 256,
  'episode_return': 5.939139423257128,
  'engine_return': 103.384,
  'terminated': False,
  'truncated': True,
  'ending_reason': 'max_episode_steps',
  'artifact_reference': '/Users/shanmukh/Desktop/Projects/AresSim/datasets/scripted-env1447-agent9001-trajectory-v4/manifest.json'}]

## Validate and read the recorded dataset

Validation checks the manifest, trajectory shards, standalone episode hashes and payloads, declared spaces, `T+1/T` alignment, reward totals, ending placement, and episode counts before the reader returns any episode. Each path under `episodes/` is a standalone `aresim.trajectory.episode.v1` file that contains both policy and replay data and can be loaded directly in the UI.

In [13]:
manifest = validate_trajectory_dataset(OUTPUT_DIRECTORY)
recorded_episodes = tuple(iter_trajectory_episodes(OUTPUT_DIRECTORY))

assert manifest.episode_count == len(rollout.episodes)
assert manifest.transition_count == rollout.transition_count
assert len(recorded_episodes) == len(rollout.episodes)

{
    "dataset": str(OUTPUT_DIRECTORY),
    "schema_version": manifest.payload["schema_version"],
    "episodes": manifest.episode_count,
    "transitions": manifest.transition_count,
    "compression": manifest.payload["compression"],
    "shards": manifest.payload["shards"],
    "trajectory_episodes": [str(OUTPUT_DIRECTORY / item["path"]) for item in manifest.payload["episode_artifacts"]],
}

{'dataset': '/Users/shanmukh/Desktop/Projects/AresSim/datasets/scripted-env1447-agent9001-trajectory-v4',
 'schema_version': 'aresim.trajectory.v1',
 'episodes': 1,
 'transitions': 256,
 'compression': 'none',
 'shards': [{'episode_count': 1,
   'path': 'episodes-00000.jsonl',
   'sha256': 'a476108e43f3c4189143cd3fa8b5e607ec7abda77c4b5cee11070110e695bf8b',
   'transition_count': 256}],
 'trajectory_episodes': ['/Users/shanmukh/Desktop/Projects/AresSim/datasets/scripted-env1447-agent9001-trajectory-v4/episodes/episode-000000.json']}

## Assert round-trip and seed determinism

The first comparison checks the JSON trajectory reader against the in-memory episode. The second runs the same policy again without recording and verifies that identical environment and policy seeds reproduce the transition sequence.

In [14]:
def assert_nested_equal(expected, actual) -> None:
    """Compare nested trajectory values while respecting NumPy arrays."""
    if isinstance(expected, np.ndarray):
        np.testing.assert_array_equal(expected, actual)
        return
    if isinstance(expected, Mapping):
        assert expected.keys() == actual.keys()
        for key in expected:
            assert_nested_equal(expected[key], actual[key])
        return
    if isinstance(expected, Sequence) and not isinstance(expected, (str, bytes)):
        assert len(expected) == len(actual)
        for expected_item, actual_item in zip(expected, actual, strict=True):
            assert_nested_equal(expected_item, actual_item)
        return
    assert expected == actual


TRANSITION_FIELDS = (
    "observations",
    "action_masks",
    "actions",
    "rewards",
    "reward_breakdowns",
    "engine_rewards",
    "engine_reward_terms",
    "terminated",
    "truncated",
    "action_legal",
    "effective_actions",
    "events",
    "terminal_reasons",
    "state_checksums",
)


def assert_episode_matches(expected, actual) -> None:
    """Assert identity metadata and every transition-aligned field."""
    for field_name in (
        "episode_id",
        "agent_id",
        "policy_id",
        "scenario_id",
        "task_id",
        "observation_schema",
        "action_schema",
        "reward_profile",
        "environment_seed",
        "agent_seed",
    ):
        assert getattr(expected, field_name) == getattr(actual, field_name)
    for field_name in TRANSITION_FIELDS:
        assert_nested_equal(getattr(expected, field_name), getattr(actual, field_name))


episode_artifact_path = OUTPUT_DIRECTORY / manifest.payload["episode_artifacts"][0]["path"]
episode_artifact = validate_trajectory_episode(episode_artifact_path)
assert episode_artifact["schemaVersion"] == "aresim.trajectory.episode.v1"
assert episode_artifact["policy"] is not None
assert episode_artifact["replay"]["schemaVersion"] == "aresim.trajectory.replay.v1"

for in_memory, recorded in zip(rollout.episodes, recorded_episodes, strict=True):
    assert_episode_matches(in_memory, recorded)
    assert_nested_equal(in_memory.replay, recorded.replay)
    assert recorded.replay["schemaVersion"] == "aresim.trajectory.replay.v1"

repeat = RolloutRunner(
    rollout_config,
    POLICY,
    environment_config=ENVIRONMENT_CONFIG,
    registry=REGISTRY,
).run()
for original, repeated in zip(rollout.episodes, repeat.episodes, strict=True):
    assert_episode_matches(original, repeated)

print("Pipeline passed: environment, policy, rollout, trajectory round-trip, and determinism.")

Pipeline passed: environment, policy, rollout, trajectory round-trip, and determinism.


## Using this notebook for a future algorithm

A future checkpoint-backed policy should implement the public `Agent` contract: versioned `policy_id`, compatible observation/action schemas, `reset(seed)`, and `act(observation, action_mask)`. Then either:

- register it under a new semantic name, assign the registry to `REGISTRY`, and set `POLICY` to that name; or
- construct it directly and assign the object to `POLICY`.

Keep the remaining cells unchanged. Passing the notebook demonstrates schema compatibility, valid action outputs, bounded collection, trajectory integrity, fixed-seed reproducibility, and UI replay export. Load any JSON file reported under `trajectory_episodes` through the UI Replay control. Each file contains the complete policy trace and replay projection in one `aresim.trajectory.episode.v1` document. RLlib retains its native online EnvRunners and learner batches; restore a native checkpoint with `aresim.training.make_checkpoint_agent(...)`, assign that agent to `POLICY`, and use this notebook for common evaluation and artifact export. Training itself is launched with `aresim-rl train`; training metrics live in W&B, while checkpoints, evaluations, trajectories, and the executed analysis notebook remain local run artifacts. `aresim-rl report` executes the checkout template `notebooks/evaluation_report_template.ipynb`. The seed-split YAML is `notebooks/phase1_open_exploration_split_v1.yaml`. Neither file is packaged inside `aresim`.